In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def show(image, title: str = None, cmap: str = None):
    plt.imshow(image, cmap=cmap)
    if title:
        plt.title(title)
    plt.axis('off')
    plt.show()

In [ ]:
from pathlib import Path


ds_path = Path('S1_v2/')


In [ ]:
from petroscope.segmentation.classes import ClassSet, LumenStoneClasses

classset = LumenStoneClasses.S1()
for cl in classset.classes:
    print(cl)

2026-03-15T14:14:46.249948+0300 INFO Loading module: torch
2026-03-15T14:14:46.347903+0300 INFO Loading module: torch.nn


[0, bg (background), color: #000000]
[1, ccp (chalcopyrite), color: #ffa500]
[2, gl (galena), color: #9acd32]
[4, br (bornite), color: #00bfff]
[6, py (pyrite), color: #2f4f4f]
[8, sh (sphalerite), color: #ee82ee]
[11, tnt (tenantite), color: #483d8b]


In [ ]:
import torch
import torch.nn as nn

def double_convolution(in_channels, out_channels):
    conv_op = nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
        nn.ReLU(inplace=True)
    )
    return conv_op

In [ ]:
import cv2
import petroscope.segmentation as segm
from pathlib import Path
from typing import Iterable
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from petroscope.segmentation.classes import ClassSet, LumenStoneClasses
from petroscope.segmentation.utils import load_image, load_mask

class SegmentationDataset(Dataset):
    def __init__(
        self,
        img_mask_paths: Iterable[tuple[Path, Path]],
        code_to_idx: dict[int, int],
        image_size: int,
    ) -> None:
        self.img_mask_paths = list(img_mask_paths)
        self.code_to_idx = code_to_idx
        self.image_size = image_size

    def __len__(self) -> int:
        return len(self.img_mask_paths)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        img_p, mask_p = self.img_mask_paths[index]
        image = load_image(img_p, normalize=True).astype(np.float32)
        mask = load_mask(mask_p)

        image = cv2.resize(
            image,
            (self.image_size, self.image_size),
            interpolation=cv2.INTER_LINEAR,
        )
        mask = cv2.resize(
            mask,
            (self.image_size, self.image_size),
            interpolation=cv2.INTER_NEAREST,
        )

        encoded_mask = np.zeros(mask.shape, dtype=np.int64)
        for code, class_idx in self.code_to_idx.items():
            encoded_mask[mask == code] = class_idx

        image_tensor = torch.from_numpy(np.transpose(image, (2, 0, 1))).float()
        mask_tensor = torch.from_numpy(encoded_mask).long()
        return image_tensor, mask_tensor


class UnetModel(segm.GeoSegmModel, nn.Module):
    def __init__(self, classes: ClassSet) -> None:
        super().__init__()
        nn.Module.__init__(self)
        self.classes = classes
        self.class_codes = [cl.code for cl in classes.classes]
        self.code_to_idx = {code: idx for idx, code in enumerate(self.class_codes)}
        self.idx_to_code = {idx: code for code, idx in self.code_to_idx.items()}
        self.code_to_color = {cl.code: cl.color for cl in classes.classes}
        self.num_classes = len(self.class_codes)
        self.max_classes = LumenStoneClasses.max_classes()
        self.image_size = 256
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print("Device: ", self.device)

        self.max_pool2d = nn.MaxPool2d(kernel_size=2, stride=2)

        self.down_convolution_1 = double_convolution(3, 64)
        self.down_convolution_2 = double_convolution(64, 128)
        self.down_convolution_3 = double_convolution(128, 256)
        self.down_convolution_4 = double_convolution(256, 512)
        self.down_convolution_5 = double_convolution(512, 1024)

        self.up_transpose_1 = nn.ConvTranspose2d(
            in_channels=1024, out_channels=512,
            kernel_size=2,
            stride=2,
        )
        self.up_convolution_1 = double_convolution(1024, 512)
        self.up_transpose_2 = nn.ConvTranspose2d(
            in_channels=512, out_channels=256,
            kernel_size=2,
            stride=2,
        )
        self.up_convolution_2 = double_convolution(512, 256)
        self.up_transpose_3 = nn.ConvTranspose2d(
            in_channels=256, out_channels=128,
            kernel_size=2,
            stride=2,
        )
        self.up_convolution_3 = double_convolution(256, 128)
        self.up_transpose_4 = nn.ConvTranspose2d(
            in_channels=128, out_channels=64,
            kernel_size=2,
            stride=2,
        )
        self.up_convolution_4 = double_convolution(128, 64)
        self.out = nn.Conv2d(
            in_channels=64, out_channels=self.num_classes,
            kernel_size=1,
        )

        self.to(self.device)

    def load(self, saved_path: Path, **kwargs) -> None:
        checkpoint = torch.load(saved_path, map_location=self.device)
        self.load_state_dict(checkpoint["state_dict"])
        self.image_size = int(checkpoint.get("image_size", self.image_size))
        loaded_code_to_idx = checkpoint.get("code_to_idx")
        if loaded_code_to_idx is not None:
            self.code_to_idx = {int(code): int(idx) for code, idx in loaded_code_to_idx.items()}
            self.idx_to_code = {idx: code for code, idx in self.code_to_idx.items()}
        loaded_code_to_color = checkpoint.get("code_to_color")
        if loaded_code_to_color is not None:
            self.code_to_color = loaded_code_to_color
        self.to(self.device)
        nn.Module.train(self, False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        down_1 = self.down_convolution_1(x)
        down_2 = self.max_pool2d(down_1)
        down_3 = self.down_convolution_2(down_2)
        down_4 = self.max_pool2d(down_3)
        down_5 = self.down_convolution_3(down_4)
        down_6 = self.max_pool2d(down_5)
        down_7 = self.down_convolution_4(down_6)
        down_8 = self.max_pool2d(down_7)
        down_9 = self.down_convolution_5(down_8)

        up_1 = self.up_transpose_1(down_9)
        x = self.up_convolution_1(torch.cat([down_7, up_1], dim=1))

        up_2 = self.up_transpose_2(x)
        x = self.up_convolution_2(torch.cat([down_5, up_2], dim=1))

        up_3 = self.up_transpose_3(x)
        x = self.up_convolution_3(torch.cat([down_3, up_3], dim=1))

        up_4 = self.up_transpose_4(x)
        x = self.up_convolution_4(torch.cat([down_1, up_4], dim=1))
        return self.out(x)

    def train(
        self, img_mask_paths: Iterable[tuple[Path, Path]], **kwargs
    ) -> None:
        epochs = int(kwargs.get("epochs", 3))
        batch_size = int(kwargs.get("batch_size", 2))
        learning_rate = float(kwargs.get("lr", 1e-3))
        image_size = int(kwargs.get("image_size", 256))
        num_workers = int(kwargs.get("num_workers", 0))
        save_path = Path(kwargs.get("save_path", "output/unet_model.pt"))

        img_mask_paths = list(img_mask_paths)
        if not img_mask_paths:
            raise ValueError("img_mask_paths must contain at least one image-mask pair")

        self.image_size = image_size
        dataset = SegmentationDataset(
            img_mask_paths=img_mask_paths,
            code_to_idx=self.code_to_idx,
            image_size=self.image_size,
        )
        loader = DataLoader(
            dataset,
            batch_size=batch_size,
            shuffle=True,
            num_workers=num_workers,
        )

        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(self.parameters(), lr=learning_rate)

        self.to(self.device)
        nn.Module.train(self, True)
        for epoch in range(epochs):
            epoch_loss = 0.0
            progress = tqdm(loader, desc=f"Epoch {epoch + 1}/{epochs}")
            for images, masks in progress:
                images = images.to(self.device)
                masks = masks.to(self.device)

                optimizer.zero_grad(set_to_none=True)
                logits = self(images)
                loss = criterion(logits, masks)
                loss.backward()
                optimizer.step()

                epoch_loss += loss.item() * images.size(0)
                progress.set_postfix(loss=f"{loss.item():.4f}")

            mean_loss = epoch_loss / len(dataset)
            print(f"Epoch {epoch + 1}/{epochs}: loss={mean_loss:.4f}")

        save_path.parent.mkdir(parents=True, exist_ok=True)
        torch.save(
            {
                "state_dict": self.state_dict(),
                "image_size": self.image_size,
                "code_to_idx": self.code_to_idx,
                "code_to_color": self.code_to_color,
            },
            save_path,
        )
        print(f"Saved model to {save_path}")
        nn.Module.train(self, False)

    def predict_image(self, image: np.ndarray) -> np.ndarray:
        original_height, original_width = image.shape[:2]
        if image.dtype == np.uint8:
            normalized_image = image.astype(np.float32) / 255.0
        else:
            normalized_image = image.astype(np.float32)
            if normalized_image.max() > 1.0:
                normalized_image /= 255.0

        resized_image = cv2.resize(
            normalized_image,
            (self.image_size, self.image_size),
            interpolation=cv2.INTER_LINEAR,
        )
        image_tensor = torch.from_numpy(
            np.transpose(resized_image, (2, 0, 1))
        ).unsqueeze(0).float().to(self.device)

        self.to(self.device)
        nn.Module.train(self, False)
        with torch.no_grad():
            logits = self(image_tensor)
            pred_indices = torch.argmax(logits, dim=1).squeeze(0).cpu().numpy()

        pred_codes = np.zeros(pred_indices.shape, dtype=np.uint8)
        for class_idx, code in self.idx_to_code.items():
            pred_codes[pred_indices == class_idx] = code

        pred_codes = cv2.resize(
            pred_codes,
            (original_width, original_height),
            interpolation=cv2.INTER_NEAREST,
        )

        prediction = np.zeros(
            (original_height, original_width, self.max_classes),
            dtype=np.float32,
        )
        ys, xs = np.indices((original_height, original_width))
        prediction[ys, xs, pred_codes] = 1.0
        return prediction

In [ ]:
# fill correct path to the dataset

train_img_mask_p = [
    (img_p, ds_path / "masks" / "train" / f"{img_p.stem}.png")
    for img_p in sorted((ds_path / "imgs" / "train").iterdir())
]
test_img_mask_p = [
    (img_p, ds_path / "masks" / "test" / f"{img_p.stem}.png")
    for img_p in sorted((ds_path / "imgs" / "test").iterdir())
]

print(train_img_mask_p)
print(test_img_mask_p)

[(WindowsPath('S1_v2/imgs/train/train_01.jpg'), WindowsPath('S1_v2/masks/train/train_01.png')), (WindowsPath('S1_v2/imgs/train/train_02.jpg'), WindowsPath('S1_v2/masks/train/train_02.png')), (WindowsPath('S1_v2/imgs/train/train_03.jpg'), WindowsPath('S1_v2/masks/train/train_03.png')), (WindowsPath('S1_v2/imgs/train/train_04.jpg'), WindowsPath('S1_v2/masks/train/train_04.png')), (WindowsPath('S1_v2/imgs/train/train_05.jpg'), WindowsPath('S1_v2/masks/train/train_05.png')), (WindowsPath('S1_v2/imgs/train/train_06.jpg'), WindowsPath('S1_v2/masks/train/train_06.png')), (WindowsPath('S1_v2/imgs/train/train_07.jpg'), WindowsPath('S1_v2/masks/train/train_07.png')), (WindowsPath('S1_v2/imgs/train/train_08.jpg'), WindowsPath('S1_v2/masks/train/train_08.png')), (WindowsPath('S1_v2/imgs/train/train_09.jpg'), WindowsPath('S1_v2/masks/train/train_09.png')), (WindowsPath('S1_v2/imgs/train/train_10.jpg'), WindowsPath('S1_v2/masks/train/train_10.png')), (WindowsPath('S1_v2/imgs/train/train_11.jpg'), Wi

In [ ]:
model = UnetModel(classes=classset)
model.train(
    img_mask_paths=train_img_mask_p[:15],
    epochs=3,
    batch_size=2,
    image_size=256,
    lr=1e-3,
    save_path=Path("output/unet_model.pt"),
)
print("Trained classes:\n", "\n".join([f"{code}: {color}" for code, color in model.code_to_color.items()]))

cpu


Epoch 1/3:   0%|          | 0/8 [00:03<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
from petroscope.segmentation.eval import SegmDetailedTester

tester = SegmDetailedTester(
    out_dir=Path("output"),
    classes=classset,
    max_classes=LumenStoneClasses.max_classes(),
    void_pad=0,
    void_border_width=4,
    vis_segmentation=True,
)
res, res_void = tester.test_on_set(
    test_img_mask_p[:3], # remove limit in future!
    predict_func=model.predict_image,
    epoch=0,
)
print("results without void borders:\n", res)
print("results with void borders:\n", res_void)

testing: 100%|██████████| 3/3 [01:41<00:00, 33.68s/it]

results without void borders:
 	 iou [soft]:
		 bg: 0.0000 [0.0000]
		 ccp: 0.0000 [0.0000]
		 gl: 0.0000 [0.0000]
		 py: 0.0057 [0.0057]
		 sh: 0.2495 [0.2495]
		 tnt: 0.0000 [0.0000]
	 mean iou [soft]: 0.0425 [0.0425]
	 acc: 0.2505

results with void borders:
 	 iou [soft]:
		 bg: 0.0000 [0.0000]
		 ccp: 0.0000 [0.0000]
		 gl: 0.0000 [0.0000]
		 py: 0.0052 [0.0052]
		 sh: 0.2472 [0.2472]
		 tnt: 0.0000 [0.0000]
	 mean iou [soft]: 0.0421 [0.0421]
	 acc: 0.2911

